In [1]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.0/828.0 kB 67.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 MB 221.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import math
import pandas as pd
import numpy as np

## Read data from S3 and join

In [ ]:
source_bucket = 's3://msds-26.2-data'
file_names = ['sanitized_2022.csv', 'sanitized_2023.csv', 'sanitized_2024.csv', 'sanitized_2025.csv']

In [ ]:
# Read csv from s3 and concat
df_recovery = pl.DataFrame()
for name in file_names:
    file_path = f'{source_bucket}/{name}'
    df_name = pl.read_csv(file_path, separator="\t")
    df_recovery = pl.concat([df_recovery, df_name])

In [ ]:
del df_name

In [ ]:
len(df_recovery)

In [ ]:
pl.Config.set_tbl_rows(50)

In [ ]:
df_recovery.sort(by=['hashed_fc', 'year', 'month', 'week', 'gl_product_group'])

## Data Cleaning

In [ ]:
# Drop columns where all values are null
cols_to_drop = (
    df_recovery.select(pl.all().is_null().all())
    .unpivot()
    .filter(pl.col("value") == True)
    .select("variable")
    .to_series()
    .to_list()
)

# Drop the identified columns from the original DataFrame
df_recovery = df_recovery.drop(cols_to_drop)

In [ ]:
# Remove all C-Returns from the data
df_recovery = df_recovery.filter(pl.col('recovery_type') != 'C-Returns')

In [ ]:
# Create target variable for propensity to waste model
df_recovery = df_recovery.with_columns(
    pl.when(pl.col('recovery_type') == 'Sales').then(pl.lit(0))
    .otherwise(pl.lit(1)).alias('recovery')
)

In [ ]:
df_recovery.filter(pl.col('country') != pl.col('marketplace_id'))

In [ ]:
# Remove marketplace_id b/c it's a duplicate of country
df_recovery = df_recovery.drop(['marketplace_id'])

In [ ]:
# Reorder the columns
df_recovery = df_recovery.select([
 'hashed_fc',
 'year',
 'month',
 'week',
 'gl_product_group',
 'product_type',
 'macro_category',
 'item_disposition_code',
 'reason_code',
 'application_name',
 'is_stranded',
 'reason_code_type',
 'reason_code_stranded',
 'stranded_potential_issue',
 'is_hazmat',
 'units',
 'cogs',
 'weight',
 'country',
 'country_state',
 'zip_code',
 'site_type',
 'site_category',
 'recovery',
 'recovery_type'])

In [ ]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data.parquet"
df_recovery.write_parquet(destination)

## Aggregation and Feature Engineering

In [4]:
df_recovery = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data.parquet")

In [5]:
# Remove 3000 isntances of missing gl_product_group
df_recovery = df_recovery.filter(pl.col('gl_product_group').is_not_null())
df_recovery = df_recovery.filter(pl.col('gl_product_group') != -1)

In [6]:
# Aggregate to the site-week-gl level
df_recovery_agg = (
    df_recovery
    .group_by(['hashed_fc', 'year', 'month', 'week', 'gl_product_group'])
    .agg(
        pl.len().alias('num_records'),

        pl.when(pl.col("macro_category") == "RETAIL")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_RETAIL"),

        pl.when(pl.col("macro_category") == "FBA")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_FBA"),

        pl.when(pl.col("is_hazmat") == "Y")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_hazmat"),

        pl.when(pl.col("product_type") == "Food")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_food"),

        pl.when(pl.col("product_type") == "Non Food")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_non_food"),

        pl.when(pl.col("product_type") == "Pet Food")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_pet_food"),

        pl.col('units').sum().alias('units_total'),
        pl.col('cogs').sum().alias('cogs_total'),
        pl.col('weight').sum().alias('weight_total'),

        # Site-level attributes (take the first value since they should be the same across the group)
        pl.first('country'),
        pl.first('country_state'),
        pl.first('zip_code'),
        pl.first('site_type'),
        pl.first('site_category'),

        # Overall recovered units (non-sales)
        pl.when(pl.col("recovery_type") != "Sales")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_recovered"),

        # Recovery type breakdowns
        pl.when(pl.col("recovery_type") == "Remove Return")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_remove_return"),

        pl.when(pl.col("recovery_type") == "Bintool Donations")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_bintool_donations"),

        pl.when(pl.col("recovery_type") == "Donations")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_donations"),

        pl.when(pl.col("recovery_type") == "Warehouse Deals and G&R")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_warehouse_deals_and_gr"),

        pl.when(pl.col("recovery_type") == "Liquidations")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_liquidations"),

        pl.when(pl.col("recovery_type") == "Sales")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_sales"),

        pl.when(pl.col("recovery_type") == "Return to Vendor")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_return_to_vendor"),

        pl.when(pl.col("recovery_type") == "Bintool Theft")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_bintool_theft"),

        pl.when(pl.col("recovery_type") == "Remove Liquidate")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_remove_liquidate"),

        pl.when(pl.col("recovery_type") == "Bintool Remove Liquidate")
        .then(pl.col("units"))
        .otherwise(0)
        .sum()
        .alias("units_bintool_remove_liquidate"),
    )
)

In [ ]:
# Derive share features
df_recovery_agg = df_recovery_agg.with_columns([
    (pl.col("units_food") / 
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_food"),

    (pl.col("units_non_food") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_non_food"),

    (pl.col("units_pet_food") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_pet_food"),

    (pl.col("units_RETAIL") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_RETAIL"),

    (pl.col("units_FBA") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_FBA"),

    (pl.col("units_hazmat") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("share_hazmat"),

    (pl.col("units_recovered") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_recovered"),

    # Recovery type proportion
    (pl.col("units_remove_return") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_remove_return"),

    (pl.col("units_bintool_donations") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_bintool_donations"),

    (pl.col("units_donations") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_donations"),

    (pl.col("units_warehouse_deals_and_gr") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_warehouse_deals_and_gr"),

    (pl.col("units_liquidations") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_liquidations"),

    (pl.col("units_sales") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_sales"),

    (pl.col("units_return_to_vendor") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_return_to_vendor"),

    (pl.col("units_bintool_theft") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_bintool_theft"),

    (pl.col("units_remove_liquidate") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_remove_liquidate"),

    (pl.col("units_bintool_remove_liquidate") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("prob_bintool_remove_liquidate"),
])

In [9]:
# Recovery type proportion by units_recovered (shares)
df_recovery_agg = df_recovery_agg.with_columns([
    (pl.col("units_remove_return") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_remove_return"),

    (pl.col("units_bintool_donations") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_bintool_donations"),

    (pl.col("units_donations") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_donations"),

    (pl.col("units_warehouse_deals_and_gr") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_warehouse_deals_and_gr"),

    (pl.col("units_liquidations") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_liquidations"),

    (pl.col("units_sales") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_sales"),

    (pl.col("units_return_to_vendor") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_return_to_vendor"),

    (pl.col("units_bintool_theft") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_bintool_theft"),

    (pl.col("units_remove_liquidate") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_remove_liquidate"),

    (pl.col("units_bintool_remove_liquidate") /
     pl.when(pl.col("units_recovered") > 0)
       .then(pl.col("units_recovered"))
       .otherwise(None)
    ).alias("share_bintool_remove_liquidate"),
])

In [10]:
# Derive cogs per unit features
df_recovery_agg = df_recovery_agg.with_columns([
    (pl.col("cogs_total") / 
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("avg_cogs_per_unit"),

    (pl.col("weight_total") /
     pl.when(pl.col("units_total") > 0)
       .then(pl.col("units_total"))
       .otherwise(None)
    ).alias("avg_weight_per_unit"),

    (pl.col("cogs_total") /
     pl.when(pl.col("weight_total") > 0)
       .then(pl.col("weight_total"))
       .otherwise(None)
    ).alias("cogs_per_unit_weight"),
])

In [11]:
# Create ISO week-date
df_recovery_agg = df_recovery_agg.with_columns(
    pl.date(pl.col("year"), 1, 4)
    .dt.truncate("1w")
    .dt.offset_by(
        (pl.col("week").cast(pl.Int32) - 1).cast(pl.Utf8) + "w"
    )
    .alias("week_date")
)

In [12]:
# Create features for total unit at site
df_recovery_agg = df_recovery_agg.with_columns([
    pl.sum("units_total").over(["hashed_fc","week_date"]).alias("site_units_total_week"),
    pl.sum("weight_total").over(["hashed_fc","week_date"]).alias("site_weight_total_week"),
    pl.sum("units_recovered").over(["hashed_fc","week_date"]).alias("site_units_recovered_week"),

    # Recovery type site totals
    pl.sum("units_remove_return").over(["hashed_fc","week_date"]).alias("site_units_remove_return_week"),
    pl.sum("units_bintool_donations").over(["hashed_fc","week_date"]).alias("site_units_bintool_donations_week"),
    pl.sum("units_donations").over(["hashed_fc","week_date"]).alias("site_units_donations_week"),
    pl.sum("units_warehouse_deals_and_gr").over(["hashed_fc","week_date"]).alias("site_units_warehouse_deals_and_gr_week"),
    pl.sum("units_liquidations").over(["hashed_fc","week_date"]).alias("site_units_liquidations_week"),
    pl.sum("units_sales").over(["hashed_fc","week_date"]).alias("site_units_sales_week"),
    pl.sum("units_return_to_vendor").over(["hashed_fc","week_date"]).alias("site_units_return_to_vendor_week"),
    pl.sum("units_bintool_theft").over(["hashed_fc","week_date"]).alias("site_units_bintool_theft_week"),
    pl.sum("units_remove_liquidate").over(["hashed_fc","week_date"]).alias("site_units_remove_liquidate_week"),
    pl.sum("units_bintool_remove_liquidate").over(["hashed_fc","week_date"]).alias("site_units_bintool_remove_liquidate_week"),
])

In [13]:
# Create share of current gl at site
df_recovery_agg = df_recovery_agg.with_columns(
    (pl.col("units_total") / pl.col("site_units_total_week"))
    .alias("site_units_share_week"),
    (pl.col("weight_total") / pl.col("site_weight_total_week"))
    .alias("site_weight_share_week"),
    (pl.col("units_recovered") / pl.col("site_units_recovered_week"))
    .alias("site_recovered_share_week"),

    # Overall recovery probability at site level
    (pl.col("site_units_recovered_week") / pl.col("site_units_total_week"))
    .alias("site_prob_recovered_week"),

    # Per recovery outcome probability at site level
    (pl.col("site_units_remove_return_week") / pl.col("site_units_total_week"))
    .alias("site_prob_remove_return_week"),
    (pl.col("site_units_bintool_donations_week") / pl.col("site_units_total_week"))
    .alias("site_prob_bintool_donations_week"),
    (pl.col("site_units_donations_week") / pl.col("site_units_total_week"))
    .alias("site_prob_donations_week"),
    (pl.col("site_units_warehouse_deals_and_gr_week") / pl.col("site_units_total_week"))
    .alias("site_prob_warehouse_deals_and_gr_week"),
    (pl.col("site_units_liquidations_week") / pl.col("site_units_total_week"))
    .alias("site_prob_liquidations_week"),
    (pl.col("site_units_sales_week") / pl.col("site_units_total_week"))
    .alias("site_prob_sales_week"),
    (pl.col("site_units_return_to_vendor_week") / pl.col("site_units_total_week"))
    .alias("site_prob_return_to_vendor_week"),
    (pl.col("site_units_bintool_theft_week") / pl.col("site_units_total_week"))
    .alias("site_prob_bintool_theft_week"),
    (pl.col("site_units_remove_liquidate_week") / pl.col("site_units_total_week"))
    .alias("site_prob_remove_liquidate_week"),
    (pl.col("site_units_bintool_remove_liquidate_week") / pl.col("site_units_total_week"))
    .alias("site_prob_bintool_remove_liquidate_week"),
)

In [14]:
df_recovery_agg = df_recovery_agg.with_columns(pl.col('site_recovered_share_week').fill_nan(0))

In [15]:
# round all numeric columns to 6 decimal places
df_recovery_agg = df_recovery_agg.with_columns(
    pl.col(pl.Float32, pl.Float64).round(6)
)

In [16]:
group_keys = ["hashed_fc", "gl_product_group"]

# Build complete week grid
full_weeks = (
    df_recovery_agg.group_by(group_keys)
      .agg([
          pl.min("week_date").alias("start"),
          pl.max("week_date").alias("end"),
      ])
      .with_columns(
          pl.date_ranges("start", "end", interval="1w").alias("week_date")
      )
      .explode("week_date")
)

# Join and Sort
df_recovery_agg_full_weeks = (
    full_weeks
    .join(df_recovery_agg, on=group_keys + ["week_date"], how="left")
    .sort(group_keys + ["week_date"])
)

In [17]:
# Calculate lag, rolling, and EWMA features for numeric columns
group_keys = ["hashed_fc", "gl_product_group"]

# Columns to exclude — identifiers, dates, and grid artifacts
exclude_cols = {
    "hashed_fc", "gl_product_group",
    "start", "end", "week_date",
    "year", "month", "week",
    "country", "country_state", "zip_code",
    "site_type", "site_category", "num_records"
}

# Auto-detect numeric feature columns
feature_cols = [
    col for col in df_recovery_agg_full_weeks.columns
    if col not in exclude_cols
    and df_recovery_agg_full_weeks[col].dtype in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64,
    )
]

lag_weeks          = [1, 4, 12, 13, 52]
rolling_weeks      = [4, 12]
rolling_weeks_long = [26, 52]
ewma_alphas        = [0.5, 0.1]

# --- Lag features ---
lag_exprs = [
    pl.col(col).shift(n).over(group_keys).alias(f"{col}_lag_{n}w")
    for col in feature_cols
    for n in lag_weeks
]

# --- Short rolling (require at least half the window) ---
rolling_exprs = [
    pl.col(col).shift(1).rolling_mean(window_size=n, min_periods=max(1, n // 2))
    .over(group_keys).alias(f"{col}_rolling_{n}w")
    for col in feature_cols
    for n in rolling_weeks
]

# --- Long rolling (lenient, min 4 periods) ---
rolling_exprs_long = [
    pl.col(col).shift(1).rolling_mean(window_size=n, min_periods=4)
    .over(group_keys).alias(f"{col}_rolling_{n}w")
    for col in feature_cols
    for n in rolling_weeks_long
]

# --- EWMA (fast: alpha=0.5, slow: alpha=0.1) ---
ewma_exprs = [
    pl.col(col).shift(1).ewm_mean(alpha=alpha, min_periods=4)
    .over(group_keys).alias(f"{col}_ewma_{str(alpha).replace('0.', '')}a")
    for col in feature_cols
    for alpha in ewma_alphas
]

# --- Apply in batches to avoid memory pressure ---
df_recovery_agg_full_weeks = (
    df_recovery_agg_full_weeks
    .with_columns(lag_exprs)
    .with_columns(rolling_exprs)
    .with_columns(rolling_exprs_long)
    .with_columns(ewma_exprs)
)

print(f"Feature cols   : {len(feature_cols)}")
print(f"Lag cols       : {len(lag_exprs)}")
print(f"Rolling cols   : {len(rolling_exprs) + len(rolling_exprs_long)}")
print(f"EWMA cols      : {len(ewma_exprs)}")
print(f"Total new cols : {len(lag_exprs) + len(rolling_exprs) + len(rolling_exprs_long) + len(ewma_exprs)}")

/tmp/ipykernel_1123/3863253679.py:38: DeprecationWarning: the argument `min_periods` for `Expr.rolling_mean` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  pl.col(col).shift(1).rolling_mean(window_size=n, min_periods=max(1, n // 2))
/tmp/ipykernel_1123/3863253679.py:46: DeprecationWarning: the argument `min_periods` for `Expr.rolling_mean` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  pl.col(col).shift(1).rolling_mean(window_size=n, min_periods=4)
/tmp/ipykernel_1123/3863253679.py:54: DeprecationWarning: the argument `min_periods` for `Expr.ewm_mean` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  pl.col(col).shift(1).ewm_mean(alpha=alpha, min_periods=4)


Feature cols   : 77
Lag cols       : 385
Rolling cols   : 308
EWMA cols      : 154
Total new cols : 847


In [18]:
# Drop buffer weeks
df_recovery_agg_full_weeks = df_recovery_agg_full_weeks.drop_nulls(subset=['week'])

In [19]:
# Create sine and cosine transformation for week
period = 52.1775
df_recovery_agg_full_weeks = df_recovery_agg_full_weeks.with_columns(
    (np.sin(2 * np.pi * pl.col('week') / period)).alias('week_sin'),
    (np.cos(2 * np.pi * pl.col('week') / period)).alias('week_cos')
)

In [20]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features.parquet"
df_recovery_agg_full_weeks.write_parquet(destination)

# Create dataset that excluded site with consistent 0 recovery and site with consistent 1 recovery

In [2]:
df_recovery_agg_full_weeks = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features.parquet")

In [3]:
df_recovery_agg_full_weeks.columns

['hashed_fc',
 'gl_product_group',
 'start',
 'end',
 'week_date',
 'year',
 'month',
 'week',
 'num_records',
 'units_RETAIL',
 'units_FBA',
 'units_hazmat',
 'units_food',
 'units_non_food',
 'units_pet_food',
 'units_total',
 'cogs_total',
 'weight_total',
 'country',
 'country_state',
 'zip_code',
 'site_type',
 'site_category',
 'units_recovered',
 'units_remove_return',
 'units_bintool_donations',
 'units_donations',
 'units_warehouse_deals_and_gr',
 'units_liquidations',
 'units_sales',
 'units_return_to_vendor',
 'units_bintool_theft',
 'units_remove_liquidate',
 'units_bintool_remove_liquidate',
 'share_food',
 'share_non_food',
 'share_pet_food',
 'share_RETAIL',
 'share_FBA',
 'share_hazmat',
 'prob_recovered',
 'prob_remove_return',
 'prob_bintool_donations',
 'prob_donations',
 'prob_warehouse_deals_and_gr',
 'prob_liquidations',
 'prob_sales',
 'prob_return_to_vendor',
 'prob_bintool_theft',
 'prob_remove_liquidate',
 'prob_bintool_remove_liquidate',
 'avg_cogs_per_unit',

In [5]:
df_recovery_stats_site = df_recovery_agg_full_weeks.group_by('hashed_fc', maintain_order=True).agg(
    pl.col('prob_recovered').mean().alias('mean_recovery_rate'),
    pl.col('prob_recovered').std().alias('std_recovery_rate')
)

In [6]:
df_recovery_stats_site.filter((pl.col('mean_recovery_rate') == 1) & (pl.col('std_recovery_rate') == 0))

hashed_fc,mean_recovery_rate,std_recovery_rate
str,f64,f64
"""001127e0e2a2129b9613ea8c475a19…",1.0,0.0
"""006ed4d35e82847ed4eb5fcaef7e03…",1.0,0.0
"""00af2af278b036a72af9af0896297c…",1.0,0.0
"""019b071b1c75729f4ada5efd7b78d2…",1.0,0.0
"""0288830b52e5f42cde86999ea57830…",1.0,0.0
…,…,…
"""fa46fbf4b2d5509806a8694b03018f…",1.0,0.0
"""fd0f119ff43582c0a95a13e45809c1…",1.0,0.0
"""fd384dfa1b30b9515e09e5a27bf831…",1.0,0.0


In [7]:
df_recovery_stats_site.filter((pl.col('mean_recovery_rate') == 0) & (pl.col('std_recovery_rate') == 0))

hashed_fc,mean_recovery_rate,std_recovery_rate
str,f64,f64
"""0000c60d3b14250deb1312b21a448a…",0.0,0.0
"""0006a543eceed85f71ee81511c9622…",0.0,0.0
"""0006eba14fee0ad3fbe18994fbd57c…",0.0,0.0
"""000d0939c30b39d0e9f1a6a29d7a26…",0.0,0.0
"""000f0429097081ebde4259bc13f1bd…",0.0,0.0
…,…,…
"""fff2a2beb3ce7735f08d5055da1238…",0.0,0.0
"""fff3d7465104aa75ee0ca31d193398…",0.0,0.0
"""fff5180fdca2dfe8e376b1a2a22365…",0.0,0.0


In [10]:
df_recovery_agg_full_weeks_excluding_full_recovery_site = df_recovery_agg_full_weeks.filter(
    ~pl.col('hashed_fc').is_in(df_recovery_stats_site.filter((pl.col('mean_recovery_rate') == 1) & (pl.col('std_recovery_rate') == 0))['hashed_fc'].unique().to_list())
)

df_recovery_agg_full_weeks_excluding_full_recovery_site.write_parquet("s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features_excluding_full_recovery_site.parquet")

In [11]:
df_recovery_agg_full_weeks_excluding_full_and_zero_recovery_site = df_recovery_agg_full_weeks_excluding_full_recovery_site.filter(
    ~pl.col('hashed_fc').is_in(df_recovery_stats_site.filter((pl.col('mean_recovery_rate') == 0) & (pl.col('std_recovery_rate') == 0))['hashed_fc'].unique().to_list())
)

In [13]:
df_recovery_agg_full_weeks_excluding_full_and_zero_recovery_site.write_parquet("s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features_excluding_full_and_zero_recovery_site.parquet")